[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyMongo and Beanie, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)

# Bulk Writes and Transactions &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's boot cell. It starts the replica set on 27017 and a second plain
`mongod` on 27018, which task 6 needs. Run it first. Each task opens its own client and closes it,
so they can be run in any order.


In [1]:
import os
import random
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("pymongo") != "4.18.1" or version("beanie") != "2.2.0":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "pymongo==4.18.1", "beanie==2.2.0"], check=True)

import pymongo
from pymongo import DeleteOne, InsertOne, ReplaceOne, UpdateOne

DBPATH = "/content/mongo" if os.path.isdir("/content") else "/tmp/guide_mongo/rs"
LOGPATH = f"{DBPATH}.log"
URI = "mongodb://127.0.0.1:27017/shop"                              # no credential, anywhere
PUBLISHED = ["jammy", "noble"]                                      # codenames MongoDB builds for


def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(timeout=2000):
    """Whether a mongod is there, asked directly rather than through topology discovery."""
    try:
        with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                                 serverSelectionTimeoutMS=timeout) as client:
            client.admin.command("ping")
            return True
    except pymongo.errors.PyMongoError:
        return False


def install_server():
    """Add MongoDB's own apt repository and install the server package. Linux only."""
    if shell("which mongod")[0] == 0:
        return "already installed"

    codename = shell("lsb_release -cs")[1]
    if codename not in PUBLISHED:                                   # an unpublished one breaks apt
        print(f"  Ubuntu '{codename}' has no MongoDB repository; using '{PUBLISHED[-1]}' instead")
        codename = PUBLISHED[-1]

    if not shell("grep -o avx /proc/cpuinfo | head -1")[1]:
        raise RuntimeError("This CPU has no AVX. Every MongoDB build since 5.0 needs it, so "
                           "neither the apt package nor the tarball will start here.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc "
          f"| {sudo}gpg --dearmor -o /usr/share/keyrings/mongodb-8.0.gpg")
    shell(f'echo "deb [signed-by=/usr/share/keyrings/mongodb-8.0.gpg] '
          f'https://repo.mongodb.org/apt/ubuntu {codename}/mongodb-org/8.0 multiverse" '
          f'| {sudo}tee /etc/apt/sources.list.d/mongodb-8.0.list')
    shell(f"{sudo}apt-get -qq update "                              # this one list file only
          f"-o Dir::Etc::sourcelist=sources.list.d/mongodb-8.0.list "
          f"-o Dir::Etc::sourceparts=-")
    code, out = shell(f"{sudo}apt-get -qq -y install mongodb-org-server")
    if shell("which mongod")[0] != 0:
        raise RuntimeError(f"mongodb-org-server did not install. apt said: {out[-400:]}")
    return f"installed from the {codename} repository"


def start_server(wait=30):
    """Start mongod with a replica set name, idempotently. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No mongod is answering on 127.0.0.1:27017. Start your own server "
                           "with --replSet rs0 and run this again: this cell only installs one "
                           "on Linux, which is what Colab runs.")

    print(" ", install_server())
    os.makedirs(DBPATH, exist_ok=True)
    code, out = shell(f"mongod --dbpath {DBPATH} --replSet rs0 --bind_ip 127.0.0.1 "
                      f"--fork --logpath {LOGPATH}")
    if code != 0:                                                   # --fork hides the reason
        print("  mongod did not start. The last lines of its log:")
        print("   ", shell(f"tail -20 {LOGPATH}")[1].replace("\n", "\n    "))
        raise RuntimeError("mongod exited. The log above says why.")

    for attempt in range(1, wait + 1):
        if answering():
            return "installed and started"
        print(f"  waiting for mongod ({attempt})")
        time.sleep(1)
    raise RuntimeError(f"mongod did not answer within {wait} seconds.")

def initiate(wait=30):
    """Make the single node a replica set, which is what transactions and migrations need."""
    with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                             serverSelectionTimeoutMS=2000) as boot:
        try:                                                        # an explicit host, not getHostName()
            boot.admin.command("replSetInitiate",
                               {"_id": "rs0", "members": [{"_id": 0, "host": "127.0.0.1:27017"}]})
        except pymongo.errors.OperationFailure as error:
            if error.code != 23:                                    # 23 is AlreadyInitialized
                raise

        for attempt in range(1, wait + 1):
            hello = boot.admin.command("hello")
            if hello.get("isWritablePrimary"):
                return f"replica set {hello['setName']}, primary"
            time.sleep(1)
    raise RuntimeError(f"No primary after {wait} seconds. The last hello was: {hello}")

SIZE = 500                                                          # Indexes and the catalog raise this
KINDS = ["laptop", "monitor", "keyboard", "mouse", "cable"]
MAKERS = ["Aster", "Belden", "Corvid", "Dalgo"]


def seed(size=None, force=False):
    """Fill shop.products and shop.reviews, once, from a fixed seed so every run agrees."""
    size = SIZE if size is None else size
    client = pymongo.MongoClient(URI, tz_aware=True)
    shop = client.get_default_database()

    if not force and shop.products.estimated_document_count() == size:
        client.close()
        return size

    shop.products.drop()
    shop.reviews.drop()
    random.seed(0)                                                  # the whole reason runs agree

    products, reviews = [], []
    for number in range(size):
        kind = KINDS[number % len(KINDS)]
        product = {
            "_id": number,
            "sku": f"{kind[:3].upper()}-{number:06d}",
            "name": f"{MAKERS[number % len(MAKERS)]} {kind} {number}",
            "maker": MAKERS[number % len(MAKERS)],
            "kind": kind,
            "price": round(random.uniform(5, 2000), 2),
            "stock": random.randint(0, 400),
            "tags": sorted(random.sample(["sale", "new", "refurbished", "bulk", "clearance"], 2)),
            "size": {"w": random.randint(5, 60), "h": random.randint(2, 40)},
        }
        products.append(product)
        for _ in range(random.randint(0, 3)):
            reviews.append({"product_id": number, "stars": random.randint(1, 5),
                            "body": f"A review of {product['name']}"})

    for start in range(0, len(products), 5000):                     # batches, not one huge insert
        shop.products.insert_many(products[start:start + 5000])
    for start in range(0, len(reviews), 5000):
        shop.reviews.insert_many(reviews[start:start + 5000])

    client.close()
    return size


def report():
    """One line naming what this notebook is running against."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        build = client.admin.command("buildInfo")["version"].split(".")[0]
        shop = client.get_default_database()
        return (f"MongoDB {build} | pymongo {version('pymongo')} | beanie {version('beanie')} "
                f"| products: {shop.products.count_documents({})}")

STANDALONE = "mongodb://127.0.0.1:27018/shop"                       # started below, no replica set


def failed(error):
    """A failure's real message, without the parts that change every run."""
    details = getattr(error, "details", None) or {}
    return f"{type(error).__name__}: {details.get('errmsg', str(error).split(', full error')[0])}"


def start_standalone(wait=30):
    """A second mongod with no --replSet, so this notebook can show what it refuses to do."""
    try:
        with pymongo.MongoClient(STANDALONE + "?directConnection=true",
                                 serverSelectionTimeoutMS=1500) as probe:
            probe.admin.command("ping")
            return "already running"
    except pymongo.errors.PyMongoError:
        pass
    if sys.platform != "linux" and shell("which mongod")[0] != 0:
        return "not available here, and the section below says what it would have shown"

    path = f"{DBPATH}-standalone"
    os.makedirs(path, exist_ok=True)
    code, out = shell(f"mongod --dbpath {path} --port 27018 --bind_ip 127.0.0.1 "
                      f"--fork --logpath {path}.log")
    if code != 0:
        print("  the standalone did not start:", shell(f"tail -5 {path}.log")[1][-200:])
        return "unavailable"
    for _ in range(wait):
        try:
            with pymongo.MongoClient(STANDALONE + "?directConnection=true",
                                     serverSelectionTimeoutMS=1500) as probe:
                probe.admin.command("ping")
                return "started on port 27018"
        except pymongo.errors.PyMongoError:
            time.sleep(1)
    return "unavailable"


def accounts():
    """Two accounts, a hundred in one and nothing in the other, put back every time."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        shop = client.get_default_database()
        shop.accounts.drop()
        shop.accounts.insert_many([{"_id": "a", "n": 100}, {"_id": "b", "n": 0}])
        return {row["_id"]: row["n"] for row in shop.accounts.find().sort("_id")}


def balances():
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        shop = client.get_default_database()
        return {row["_id"]: row["n"] for row in shop.accounts.find().sort("_id")}


print("server:    ", start_server())
print("replica:   ", initiate())
print("standalone:", start_standalone())
print("seeded:    ", seed(), "products")
print("accounts:  ", accounts())
print(report())


server:     already running
replica:    replica set rs0, primary
standalone: already running
seeded:     500 products
accounts:   {'a': 100, 'b': 0}
MongoDB 8 | pymongo 4.18.1 | beanie 2.2.0 | products: 500


**1.** Three kinds of write, one request.


In [2]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()
shop.answers.drop()
shop.answers.insert_many([{"_id": 1, "n": 1}, {"_id": 2, "n": 2}])

result = shop.answers.bulk_write([
    InsertOne({"_id": 3, "n": 3}),
    UpdateOne({"_id": 1}, {"$inc": {"n": 100}}),
    DeleteOne({"_id": 2}),
])

print("inserted", result.inserted_count, "| modified", result.modified_count,
      "| deleted", result.deleted_count)
print("left:", list(shop.answers.find().sort("_id")))
client.close()


inserted 1 | modified 1 | deleted 1
left: [{'_id': 1, 'n': 101}, {'_id': 3, 'n': 3}]


One round trip for all three, and a separate count for each kind. Against a server on a network that
is one wait instead of three.


**2.** The same batch, both ways.


In [3]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()
batch = [InsertOne({"_id": number}) for number in (1, 2, 2, 3, 4)]

for ordered in (True, False):
    shop.answers.drop()
    try:
        shop.answers.bulk_write(batch, ordered=ordered)
    except pymongo.errors.BulkWriteError:
        pass
    print(f"ordered={ordered!s:5} wrote", sorted(row["_id"] for row in shop.answers.find()))
client.close()


ordered=True  wrote [1, 2]
ordered=False wrote [1, 2, 3, 4]


Ordered stopped at the duplicate and never attempted 3 or 4. Unordered attempted everything and
wrote all four distinct ids. Neither is a transaction, and both left the collection changed.


**3.** Which one failed.


In [4]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()
shop.answers.drop()

try:
    shop.answers.bulk_write([InsertOne({"_id": number}) for number in (1, 2, 2, 3, 3)],
                            ordered=False)
except pymongo.errors.BulkWriteError as error:
    print("inserted:", error.details["nInserted"])
    for problem in error.details["writeErrors"]:
        print("  index", problem["index"], "code", problem["code"])
client.close()


inserted: 3
  index 2 code 11000
  index 4 code 11000


The `index` is the position in the list you sent, which is what lets an importer report "row 4812 of
the file" rather than "something went wrong".


**4.** What each way of emptying leaves.


In [5]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()

shop.answers.drop()
shop.answers.insert_many([{"n": number} for number in range(100)])
shop.answers.create_index("n")
print("indexes:            ", [index["name"] for index in shop.answers.list_indexes()])

shop.answers.delete_many({})
print("after delete_many({}):", [index["name"] for index in shop.answers.list_indexes()])

shop.answers.insert_many([{"n": number} for number in range(100)])
shop.answers.drop()
print("after drop():        ", [index["name"] for index in shop.answers.list_indexes()])
client.close()


indexes:             ['_id_', 'n_1']
after delete_many({}): ['_id_', 'n_1']
after drop():         []


`drop()` took the index with it. The next write recreates the collection with only its `_id` index,
and every query that depended on `n` is a collection scan until somebody notices.


**5.** Ten, atomically.


In [6]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()
print("before:", accounts())


def move_ten(session):
    shop.accounts.update_one({"_id": "a"}, {"$inc": {"n": -10}}, session=session)
    shop.accounts.update_one({"_id": "b"}, {"$inc": {"n": 10}}, session=session)


with client.start_session() as session:
    session.with_transaction(move_ten)

print("after: ", balances())
client.close()


before: {'a': 100, 'b': 0}
after:  {'a': 90, 'b': 10}


Both writes name the session, so both are in the transaction. Leaving `session=` off either one
would make that write ordinary, and an abort would leave the accounts disagreeing.


**6.** The same thing against a plain mongod.


In [7]:
standalone = pymongo.MongoClient(STANDALONE, tz_aware=True)
plain = standalone.get_default_database()
plain.accounts.replace_one({"_id": "a"}, {"_id": "a", "n": 100}, upsert=True)

print("ordinary writes are fine:", plain.accounts.count_documents({}))
try:
    with standalone.start_session() as session:
        with session.start_transaction():
            plain.accounts.update_one({"_id": "a"}, {"$inc": {"n": -10}}, session=session)
except pymongo.errors.OperationFailure as error:
    print("transactions are not:", failed(error))

standalone.close()


ordinary writes are fine: 1
transactions are not: OperationFailure: Transaction numbers are only allowed on a replica set member or mongos


The server works perfectly for everything else. Transactions are recorded in the replica set oplog,
and a server with no replica set has no oplog to record them in.


---

&#8592; **Back to:** [Bulk Writes and Transactions](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/07-bulk-writes-and-transactions.ipynb)  &nbsp;&middot;&nbsp;  [PyMongo and Beanie, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)
